# Import

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from model_module import CGCLR
from trainer_module import training
import time
from scipy import stats
from mpl_toolkits import mplot3d
import warnings
warnings.filterwarnings("ignore")

# Synthentic Data Setting

In [ ]:
np.random.seed(0)
n = 50  # number of samples for each region
noise_std = 0.1

# covariates sampling
def sample_in_region(n, condition, x_range=(-1, 1), y_range=(-2, 2)):
    samples = []
    while len(samples) < n:
        x1 = np.random.uniform(x_range[0], x_range[1])
        x2 = np.random.uniform(y_range[0], y_range[1])
        if condition(x1, x2):
            samples.append((x1, x2))
    samples = np.array(samples)
    return samples[:, 0], samples[:, 1]

x1_north, x2_north = sample_in_region(n, lambda x1, x2: x2 > abs(x1))
x1_west, x2_west = sample_in_region(n, lambda x1, x2: x1 < -abs(x2))
x1_south, x2_south = sample_in_region(n, lambda x1, x2: x2 < -abs(x1))
x1_east, x2_east = sample_in_region(n, lambda x1, x2: x1 > abs(x2))

# customized clusterwise linear function
def f1(x1, x2):
    return x1+x2-10
def f2(x1, x2):
    return -x1 -x2 -3
def f3(x1, x2):
    return x1 + 2*x2 +2

# generate responses
y_north = f1(x1_north, x2_north) + np.random.normal(0, noise_std, n)
y_west  = f2(x1_west, x2_west) + np.random.normal(0, noise_std, n)
y_south = f1(x1_south, x2_south) + np.random.normal(0, noise_std, n)
y_east  = f3(x1_east, x2_east) + np.random.normal(0, noise_std, n)

# sampling results
x1_all = np.concatenate([x1_north, x1_west, x1_south, x1_east])
x2_all = np.concatenate([x2_north, x2_west, x2_south, x2_east])
y_all = np.concatenate([y_north, y_west, y_south, y_east])
dataset = np.column_stack((x1_all, x2_all, y_all))
df = pd.DataFrame(dataset, columns=["x1", "x2", "y"])
X = np.array(df[['x1','x2']])
Y = np.array(df[['y']])

# Train Dataset Preprocessing
train_X, train_Y = X, Y
scaler_x = StandardScaler()
train_X = scaler_x.fit_transform(train_X)
scaler_y = StandardScaler()
train_Y = scaler_y.fit_transform(train_Y)
train_X = torch.tensor(np.array(train_X)).reshape(-1, train_X.shape[1]).float()
train_Y = torch.tensor(np.array(train_Y)).reshape(-1, 1).float()        

In [ ]:
train_X_np = scaler_x.inverse_transform(train_X.cpu().numpy())  # shape: (4*n, 2)
train_Y_np = scaler_y.inverse_transform(train_Y.cpu().numpy()).flatten()  # shape: (4*n,)

# evaluation covariates
x1_min, x1_max = train_X_np[:, 0].min(), train_X_np[:, 0].max()
x2_min, x2_max = train_X_np[:, 1].min(), train_X_np[:, 1].max()
num_grid = 1000
x1_range = np.linspace(x1_min, x1_max, num_grid)
x2_range = np.linspace(x2_min, x2_max, num_grid)
x1_mesh, x2_mesh = np.meshgrid(x1_range, x2_range)
grid_points = np.c_[x1_mesh.ravel(), x2_mesh.ravel()]

# denoised response
def ground_truth_function(x1, x2):
    cond_north = x2 > np.abs(x1)
    cond_west  = x1 < -np.abs(x2)
    cond_south = x2 < -np.abs(x1)
    cond_east  = x1 > np.abs(x2)
    return np.select([cond_north, cond_west, cond_south, cond_east],
                     [f1(x1,x2), f2(x1,x2), f1(x1,x2), f3(x1,x2)],
                     default=np.nan)

def ground_truth_weight_function(x1, x2):
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    cond_north = x2 >  np.abs(x1)
    cond_west  = x1 < -np.abs(x2)
    cond_south = x2 < -np.abs(x1)
    cond_east  = x1 >  np.abs(x2)
    conds = [cond_north, cond_west, cond_south, cond_east]
    b   = np.select(conds, [ 1, -1,  1,  1], default=np.nan)
    w1  = np.select(conds, [ 1, -1,  1,  2], default=np.nan)
    w2  = np.select(conds, [-10, -3,-10,  2], default=np.nan)
    return np.stack([b, w1, w2], axis=-1)

Y_true_ground = ground_truth_function(x1_mesh, x2_mesh)

# 2D Contour Map with training samples
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, Y_true_ground, levels=100, cmap='viridis', alpha=0.8, vmin=-13, vmax=5)
plt.scatter(X[:, 0], X[:, 1], color='black', marker='x', s=30, label='Train Data')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("Ground Truth Function with training samples")
plt.legend(loc='center left')
plt.colorbar(contour, label="$y$")
plt.savefig('./img/True_Function.png', dpi = 500)
plt.show()

# 3D surface plot
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(111,projection='3d')
ax.view_init(45)
surface = ax.contour3D(x1_mesh, x2_mesh, Y_true_ground,500,cmap='viridis')
ax.set_xlabel("$x^1$")
ax.set_ylabel("$x^2$")
ax.set_zlabel("$y$")
fig.colorbar(surface, shrink=0.5, aspect=5, label="$y$")
plt.legend()
plt.savefig('./img/True_Function_Surface.png', dpi=500)
plt.show()

# CG-CLR (F-test with Prediction plot)

In [ ]:
# H0 Baseline: K=1

device = 'cuda' if torch.cuda.is_available() else 'cpu'

p_dim = train_X.shape[1]
num_K = 1

model = CGCLR(
    input_dim=p_dim,
    expert_num= num_K,
    output_dim=1,
    proxy_hidden_shape=[128,128,128],
    dropout = 0.0,
    device = device
)

pre_time = time.time()
best_model = training(
    model = model,
    train_X = train_X,
    train_Y = train_Y,
    val_X = train_X,
    val_Y = train_Y,
    max_epochs = 10000,
    patience = 1000,
    lr = 1e-3,
    batch_size = train_X.shape[0],
    y_scaler = scaler_y,
    device = device,
    LAMBDA = 1,
    verbose = False
)
print("Training Time(s): ",int(time.time() - pre_time))

best_model.eval()
with torch.no_grad():
    _, _, _, y_tilde = best_model(train_X.to(device))
if scaler_y:
    TRAIN_SSE = np.sum((scaler_y.inverse_transform(y_tilde.detach().cpu().numpy()) - scaler_y.inverse_transform(np.array(train_Y)))**2)
else:
    TRAIN_SSE = torch.sum((y_tilde - train_Y)**2).detach().cpu().numpy()
H0_SSE = TRAIN_SSE


# Inference for plot
grid_points_scaled = scaler_x.transform(grid_points)
grid_points_scaled = torch.tensor(grid_points_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    w_hat, w_tilde, _, y_tilde_grid = best_model(grid_points_scaled)
Y_pred = y_tilde_grid.cpu().numpy().reshape(x1_mesh.shape)
Y_pred_orig = scaler_y.inverse_transform(Y_pred)
Y_pred_proxy = (torch.concat([grid_points_scaled.cpu(),torch.ones(size=(len(grid_points_scaled),1))],axis=1) * w_hat.cpu()).sum(1)
Y_pred_proxy = Y_pred_proxy.cpu().numpy().reshape(x1_mesh.shape)
Y_pred_proxy_orig = scaler_y.inverse_transform(Y_pred_proxy)
Y_true_ground = ground_truth_function(x1_mesh, x2_mesh)
Y_pred_err_orig = np.abs(Y_pred_orig - Y_true_ground)
Y_pred_proxy_err_orig = np.abs(Y_pred_proxy_orig - Y_true_ground)
weight_true_ground = ground_truth_weight_function(x1_mesh, x2_mesh)
sigma_y = scaler_y.scale_[0]
mu_y    = scaler_y.mean_[0]
sigma_x = scaler_x.scale_        
mu_x    = scaler_x.mean_         
w       = w_tilde[:, :-1].cpu().detach().numpy()    
w0      = w_tilde[:,-1].cpu().detach().numpy()
beta_tilde    = (sigma_y * w) / sigma_x       
beta_0  = mu_y + sigma_y * w0 - np.dot(beta_tilde, mu_x)
beta_0_tilde = beta_0.reshape(-1,1)
weight_err = ((np.concatenate([beta_tilde, beta_0_tilde],axis=1).reshape(1000,1000,3) - weight_true_ground)**2).mean(2)**0.5
w       = w_hat[:, :-1].cpu().detach().numpy()    
w0      = w_hat[:,-1].cpu().detach().numpy()
beta_hat    = (sigma_y * w) / sigma_x      
beta_0  = mu_y + sigma_y * w0 - np.dot(beta_hat, mu_x)
beta_0_hat = beta_0.reshape(-1,1)
weight_proxy_err = ((np.concatenate([beta_hat, beta_0_hat],axis=1).reshape(1000,1000,3) - weight_true_ground)**2).mean(2)**0.5

with torch.no_grad():
    _, _, cluster_indices, y_tilde_train = best_model(train_X.to('cuda'))
cluster_indices = cluster_indices.detach().cpu().numpy()

# Codebook Prediction Contour Plot
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_orig, levels=100, cmap='viridis', alpha=0.8, vmin=-13, vmax=5)
color_list = ['g','b','r','brown','yellow','black']
for i in range(num_K):
    plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("CG-CLR(Codebook) Prediction Function")
plt.legend()
plt.colorbar(contour, label="$\\boldsymbol{x}_i^{\\top}\\tilde{w}_{z_i}$")
plt.savefig(f'./img/CARVE(Codebook)_Pred_K{num_K}.png', dpi = 500)
plt.show()

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(x1_mesh, x2_mesh, Y_pred_orig, cmap='viridis', edgecolor='none', alpha=0.8)
ax.scatter(train_X_np[:, 0], train_X_np[:, 1], train_Y_np, color='r', marker='x',
           s=30, label='Training Data', depthshade=True)
ax.set_xlabel("$x^1$")
ax.set_ylabel("$x^2$")
ax.set_zlabel("$y$")
ax.set_title("3D Surface Plot of CG-CLR(Codebook) Predictions")
fig.colorbar(surface, shrink=0.5, aspect=5, label="Predicted y")
plt.legend()
plt.savefig(f'./img/CARVE(Codebook)_Pred_K{num_K}_3D.png', dpi = 500)
plt.show()

# Codebook Prediction Error Plot
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_err_orig, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=2)
color_list = ['g','b','r','brown','yellow','black']
for i in range(num_K):
    plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("CG-CLR(Codebook) Prediction Error")
plt.legend()
plt.colorbar(contour, label="$|\\boldsymbol{x}_i^{\\!\\top}\\tilde{w}_{z_i} - y_i|$")
plt.savefig(f'./img/CARVE(Codebook)_Error_K{num_K}.png', dpi = 500)
plt.show()

# Codebook Recovery Error Plot
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, weight_err, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=4)
color_list = ['g','b','r','brown','yellow','black']
for i in range(num_K):
    plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("CG-CLR(Codebook) Coefficient Recovery Error")
plt.legend()
plt.colorbar(contour, label="$||\\tilde{w}_{z_i} - w^*_i||$")
plt.savefig(f'./img/CARVE(Codebook)_Recovery_Error_K{num_K}.png', dpi = 500)
plt.show()


# Proxy Prediction Contour Plot
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_proxy_orig, levels=100, cmap='viridis', alpha=0.8, vmin=-13, vmax=5)
color_list = ['g','b','r','brown','yellow','black']
for i in range(num_K):
    plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("CG-CLR(Proxy) Prediction Function")
plt.legend()
plt.colorbar(contour, label="$\\boldsymbol{x}_i^{\\top}\\tilde{w}_{z_i}$")
plt.savefig(f'./img/CARVE(Proxy)_Pred_K{num_K}.png', dpi = 500)
plt.show()

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(x1_mesh, x2_mesh, Y_pred_proxy_orig, cmap='viridis', edgecolor='none', alpha=0.8)
ax.scatter(train_X_np[:, 0], train_X_np[:, 1], train_Y_np, color='r', marker='x',
           s=30, label='Training Data', depthshade=True)
ax.set_xlabel("$x^1$")
ax.set_ylabel("$x^2$")
ax.set_zlabel("$y$")
ax.set_title("3D Surface Plot of CG-CLR(Proxy) Predictions")
fig.colorbar(surface, shrink=0.5, aspect=5, label="Predicted y")
plt.legend()
plt.savefig(f'./img/CARVE(Proxy)_Pred_K{num_K}_3D.png', dpi = 500)
plt.show()

# Proxy Prediction Error Plot
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_proxy_err_orig, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=2)
color_list = ['g','b','r','brown','yellow','black']
for i in range(num_K):
    plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("CG-CLR(Proxy) Prediction Error")
plt.legend()
plt.colorbar(contour, label="$|\\boldsymbol{x}_i^{\\!\\top}\\hat{w}_i - y_i|$")
plt.savefig(f'./img/CARVE(Proxy)_Error_K{num_K}.png', dpi = 500)
plt.show()

# Proxy Recovery Error Plot
plt.figure(figsize=(10, 8))
contour = plt.contourf(x1_mesh, x2_mesh, weight_proxy_err, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=4)
color_list = ['g','b','r','brown','yellow','black']
for i in range(num_K):
    plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
plt.xlabel("$x^1$")
plt.ylabel("$x^2$")
plt.title("CG-CLR(Proxy) Coefficient Recovery Error")
plt.legend()
plt.colorbar(contour, label="$||\\hat{w}_i - w^*_i||$")
plt.savefig(f'./img/CARVE(Proxy)_Recovery_Error_K{num_K}.png', dpi = 500)
plt.show()

In [ ]:
for num_K in range(2, 50):
    # H1 modeling
    model = CGCLR(
        input_dim=p_dim,
        expert_num= num_K,
        output_dim=1,
        proxy_hidden_shape=[128,128,128],
        dropout = 0.0,
        device = device
    )
    pre_time = time.time()
    best_model = training(
        model = model,
        train_X = train_X,
        train_Y = train_Y,
        val_X = train_X,
        val_Y = train_Y,
        max_epochs = 10000,
        patience = 1000,
        lr = 1e-3,
        batch_size = train_X.shape[0],
        y_scaler = scaler_y,
        device = device,
        LAMBDA = 1,
        verbose = False
    )
    print("Training Time(s): ",int(time.time() - pre_time))
    
    best_model.eval()
    with torch.no_grad():
        _, _, _, y_tilde = best_model(train_X.to(device))
    if scaler_y:
        TRAIN_SSE = np.sum((scaler_y.inverse_transform(y_tilde.detach().cpu().numpy()) - scaler_y.inverse_transform(np.array(train_Y)))**2)
    else:
        TRAIN_SSE = torch.sum((y_tilde - train_Y)**2)

    H1_SSE = TRAIN_SSE
    f_stat = (train_X.shape[0] - (num_K)*(p_dim+1))/(p_dim+1) * (H0_SSE / H1_SSE - 1)
    p_val = 2 * stats.f.sf(f_stat, p_dim+1, (len(train_X) - (num_K)*(p_dim+1)))
    print(f"F={f_stat}, P_val={p_val}, H0: K={num_K-1}")
    H0_SSE = H1_SSE
    
    
    # Inference for plot
    grid_points_scaled = scaler_x.transform(grid_points)
    grid_points_scaled = torch.tensor(grid_points_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        w_hat, w_tilde, _, y_tilde_grid = best_model(grid_points_scaled)
    Y_pred = y_tilde_grid.cpu().numpy().reshape(x1_mesh.shape)
    Y_pred_orig = scaler_y.inverse_transform(Y_pred)
    Y_pred_proxy = (torch.concat([grid_points_scaled.cpu(),torch.ones(size=(len(grid_points_scaled),1))],axis=1) * w_hat.cpu()).sum(1)
    Y_pred_proxy = Y_pred_proxy.cpu().numpy().reshape(x1_mesh.shape)
    Y_pred_proxy_orig = scaler_y.inverse_transform(Y_pred_proxy)
    Y_true_ground = ground_truth_function(x1_mesh, x2_mesh)
    Y_pred_err_orig = np.abs(Y_pred_orig - Y_true_ground)
    Y_pred_proxy_err_orig = np.abs(Y_pred_proxy_orig - Y_true_ground)
    weight_true_ground = ground_truth_weight_function(x1_mesh, x2_mesh)
    sigma_y = scaler_y.scale_[0]
    mu_y    = scaler_y.mean_[0]
    sigma_x = scaler_x.scale_        
    mu_x    = scaler_x.mean_         
    w       = w_tilde[:, :-1].cpu().detach().numpy()    
    w0      = w_tilde[:,-1].cpu().detach().numpy()
    beta_tilde    = (sigma_y * w) / sigma_x       
    beta_0  = mu_y + sigma_y * w0 - np.dot(beta_tilde, mu_x)
    beta_0_tilde = beta_0.reshape(-1,1)
    weight_err = ((np.concatenate([beta_tilde, beta_0_tilde],axis=1).reshape(1000,1000,3) - weight_true_ground)**2).mean(2)**0.5
    w       = w_hat[:, :-1].cpu().detach().numpy()    
    w0      = w_hat[:,-1].cpu().detach().numpy()
    beta_hat    = (sigma_y * w) / sigma_x      
    beta_0  = mu_y + sigma_y * w0 - np.dot(beta_hat, mu_x)
    beta_0_hat = beta_0.reshape(-1,1)
    weight_proxy_err = ((np.concatenate([beta_hat, beta_0_hat],axis=1).reshape(1000,1000,3) - weight_true_ground)**2).mean(2)**0.5

    with torch.no_grad():
        _, _, cluster_indices, y_tilde_train = best_model(train_X.to('cuda'))
    cluster_indices = cluster_indices.detach().cpu().numpy()

    # Codebook Prediction Contour Plot
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_orig, levels=100, cmap='viridis', alpha=0.8, vmin=-13, vmax=5)
    color_list = ['g','b','r','brown','yellow','black']
    for i in range(num_K):
        plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
    plt.xlabel("$x^1$")
    plt.ylabel("$x^2$")
    plt.title("CG-CLR(Codebook) Prediction Function")
    plt.legend()
    plt.colorbar(contour, label="$\\boldsymbol{x}_i^{\\top}\\tilde{w}_{z_i}$")
    plt.savefig(f'./img/CARVE(Codebook)_Pred_K{num_K}.png', dpi = 500)
    plt.show()

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    surface = ax.plot_surface(x1_mesh, x2_mesh, Y_pred_orig, cmap='viridis', edgecolor='none', alpha=0.8)
    ax.scatter(train_X_np[:, 0], train_X_np[:, 1], train_Y_np, color='r', marker='x',
            s=30, label='Training Data', depthshade=True)
    ax.set_xlabel("$x^1$")
    ax.set_ylabel("$x^2$")
    ax.set_zlabel("$y$")
    ax.set_title("3D Surface Plot of CG-CLR(Codebook) Predictions")
    fig.colorbar(surface, shrink=0.5, aspect=5, label="Predicted y")
    plt.legend()
    plt.savefig(f'./img/CARVE(Codebook)_Pred_K{num_K}_3D.png', dpi = 500)
    plt.show()

    # Codebook Prediction Error Plot
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_err_orig, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=2)
    color_list = ['g','b','r','brown','yellow','black']
    for i in range(num_K):
        plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
    plt.xlabel("$x^1$")
    plt.ylabel("$x^2$")
    plt.title("CG-CLR(Codebook) Prediction Error")
    plt.legend()
    plt.colorbar(contour, label="$|\\boldsymbol{x}_i^{\\!\\top}\\tilde{w}_{z_i} - y_i|$")
    plt.savefig(f'./img/CARVE(Codebook)_Error_K{num_K}.png', dpi = 500)
    plt.show()

    # Codebook Recovery Error Plot
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(x1_mesh, x2_mesh, weight_err, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=4)
    color_list = ['g','b','r','brown','yellow','black']
    for i in range(num_K):
        plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
    plt.xlabel("$x^1$")
    plt.ylabel("$x^2$")
    plt.title("CG-CLR(Codebook) Coefficient Recovery Error")
    plt.legend()
    plt.colorbar(contour, label="$||\\tilde{w}_{z_i} - w^*_i||$")
    plt.savefig(f'./img/CARVE(Codebook)_Recovery_Error_K{num_K}.png', dpi = 500)
    plt.show()


    # Proxy Prediction Contour Plot
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_proxy_orig, levels=100, cmap='viridis', alpha=0.8, vmin=-13, vmax=5)
    color_list = ['g','b','r','brown','yellow','black']
    for i in range(num_K):
        plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
    plt.xlabel("$x^1$")
    plt.ylabel("$x^2$")
    plt.title("CG-CLR(Proxy) Prediction Function")
    plt.legend()
    plt.colorbar(contour, label="$\\boldsymbol{x}_i^{\\top}\\tilde{w}_{z_i}$")
    plt.savefig(f'./img/CARVE(Proxy)_Pred_K{num_K}.png', dpi = 500)
    plt.show()

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    surface = ax.plot_surface(x1_mesh, x2_mesh, Y_pred_proxy_orig, cmap='viridis', edgecolor='none', alpha=0.8)
    ax.scatter(train_X_np[:, 0], train_X_np[:, 1], train_Y_np, color='r', marker='x',
            s=30, label='Training Data', depthshade=True)
    ax.set_xlabel("$x^1$")
    ax.set_ylabel("$x^2$")
    ax.set_zlabel("$y$")
    ax.set_title("3D Surface Plot of CG-CLR(Proxy) Predictions")
    fig.colorbar(surface, shrink=0.5, aspect=5, label="Predicted y")
    plt.legend()
    plt.savefig(f'./img/CARVE(Proxy)_Pred_K{num_K}_3D.png', dpi = 500)
    plt.show()

    # Proxy Prediction Error Plot
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(x1_mesh, x2_mesh, Y_pred_proxy_err_orig, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=2)
    color_list = ['g','b','r','brown','yellow','black']
    for i in range(num_K):
        plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
    plt.xlabel("$x^1$")
    plt.ylabel("$x^2$")
    plt.title("CG-CLR(Proxy) Prediction Error")
    plt.legend()
    plt.colorbar(contour, label="$|\\boldsymbol{x}_i^{\\!\\top}\\hat{w}_i - y_i|$")
    plt.savefig(f'./img/CARVE(Proxy)_Error_K{num_K}.png', dpi = 500)
    plt.show()

    # Proxy Prediction Error Plot
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(x1_mesh, x2_mesh, weight_proxy_err, levels=100, cmap='viridis', alpha=0.8, vmin=0, vmax=4)
    color_list = ['g','b','r','brown','yellow','black']
    for i in range(num_K):
        plt.scatter(train_X_np[(cluster_indices==i).flatten()][:, 0], train_X_np[(cluster_indices==i).flatten()][:, 1], color=color_list[i], marker='x', s=30, label=f'cluster-{i}')
    plt.xlabel("$x^1$")
    plt.ylabel("$x^2$")
    plt.title("CG-CLR(Proxy) Coefficient Recovery Error")
    plt.legend()
    plt.colorbar(contour, label="$||\\hat{w}_i - w^*_i||$")
    plt.savefig(f'./img/CARVE(Proxy)_Recovery_Error_K{num_K}.png', dpi = 500)
    plt.show()
    
    
    
    if p_val > 0.01:
        print(f"Cannot Reject H0, Thus K={num_K-1}")
        break
    

##